<a href="https://colab.research.google.com/github/Kiranwaghmare123/ADS-with-Python/blob/main/Generative_ai_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -U openai python-dotenv ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 16.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.3 MB/s  0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ipywidgets]


In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI
try:
    from google.colab import userdata
    colab_key = userdata.get("OPENAI_API_KEY")
except Exception:
    colab_key = None

load_dotenv()

API_KEY = colab_key or os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

if not API_KEY:
    raise ValueError(
        "API key not found. Please add 'OPENAI_API_KEY' to your Colab Secrets (key icon on the left panel) and enable Notebook access, or create a .env file beside your notebook."
    )

client = OpenAI(api_key=API_KEY)

print("OpenAI client initialized.")
print("Configured model:", MODEL)

OpenAI client initialized.
Configured model: gpt-4o-mini


In [16]:
try:
    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful AI assistant. Keep answers clear and concise.",
        input="Explain Generative AI in three simple bullet points."
    )
    print(response.output_text)
except Exception as e:
    print(f"API Request failed: {e}")
    print("\nTip: If you see a 429 RateLimitError regarding credits, please check your billing details at: https://platform.openai.com/settings/organization/billing")

API Request failed: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.

Tip: If you see a 429 RateLimitError regarding credits, please check your billing details at: https://platform.openai.com/settings/organization/billing


In [6]:
# Conversation memory for this notebook session
chat_history = []

SYSTEM_PROMPT = """
You are a helpful Generative AI assistant.
Explain technical concepts in simple language.
Use examples and Python code when appropriate.
If you are unsure about an answer, say so.
"""

def chat(user_message, reset=False):
    """
    Send a message to the language model and retain conversation history.

    Parameters:
        user_message (str): Message from the user.
        reset (bool): Clear the conversation before answering.

    Returns:
        str: Generated assistant response.
    """
    global chat_history

    if reset:
        chat_history = []

    if not isinstance(user_message, str) or not user_message.strip():
        return "Please enter a message."

    # Add the new user message
    chat_history.append({
        "role": "user",
        "content": user_message.strip()
    })

    try:
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_PROMPT,
            input=chat_history
        )

        answer = response.output_text

        if not answer:
            answer = "The model returned an empty response."

        # Retain the answer for the next turn
        chat_history.append({
            "role": "assistant",
            "content": answer
        })

        return answer

    except Exception as error:
        # Remove the failed user turn so it can be retried.
        chat_history.pop()
        print("API error:", str(error))
        return "Sorry, the request failed. Check your API configuration and try again."

In [17]:
answer = chat("What is a Large Language Model?")
print(answer)

API error: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.
Sorry, the request failed. Check your API configuration and try again.


In [18]:
answer = chat("Give me a simple Python example of using an LLM.")
print(answer)

API error: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.
Sorry, the request failed. Check your API configuration and try again.


In [9]:
answer = chat("Remember this project name: Smart Study Assistant.")
print(answer)

answer = chat("What project name did I tell you?")
print(answer)

API error: name 'client' is not defined
Sorry, the request failed. Check your API configuration and try again.
API error: name 'client' is not defined
Sorry, the request failed. Check your API configuration and try again.


In [10]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# UI components
title = widgets.HTML(
    value="<h2>🤖 Generative AI Chatbot</h2>"
          "<p>Ask questions and explore Generative AI.</p>"
)

message_input = widgets.Textarea(
    placeholder="Type your question here...",
    description="Message:",
    layout=widgets.Layout(width="100%", height="90px")
)

send_button = widgets.Button(
    description="Send",
    button_style="primary",
    icon="paper-plane"
)

clear_button = widgets.Button(
    description="Clear Chat",
    button_style="warning",
    icon="trash"
)

chat_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #cccccc",
        padding="12px",
        width="100%"
    )
)

def send_message(_):
    message = message_input.value.strip()

    if not message:
        return

    message_input.value = ""
    send_button.disabled = True

    try:
        with chat_output:
            display(Markdown(f"**You:** {message}"))

            answer = chat(message)

            display(Markdown(f"**AI Assistant:**\n\n{answer}"))
            display(Markdown("---"))

    finally:
        send_button.disabled = False

def clear_chat(_):
    global chat_history
    chat_history = []
    chat_output.clear_output()
    message_input.value = ""
    print("Conversation cleared.")

send_button.on_click(send_message)
clear_button.on_click(clear_chat)

display(
    widgets.VBox([
        title,
        chat_output,
        message_input,
        widgets.HBox([send_button, clear_button])
    ])
)

In [11]:
import ipywidgets as widgets
from IPython.display import display

prompt_input = widgets.Textarea(
    value=SYSTEM_PROMPT,
    description="Instructions:",
    layout=widgets.Layout(width="100%", height="150px")
)

save_prompt_button = widgets.Button(
    description="Update Instructions",
    button_style="success"
)

prompt_status = widgets.Output()

def update_prompt(_):
    global SYSTEM_PROMPT

    if not prompt_input.value.strip():
        with prompt_status:
            prompt_status.clear_output()
            print("Instructions cannot be empty.")
        return

    SYSTEM_PROMPT = prompt_input.value.strip()

    with prompt_status:
        prompt_status.clear_output()
        print("Assistant instructions updated.")

save_prompt_button.on_click(update_prompt)

display(widgets.VBox([
    prompt_input,
    save_prompt_button,
    prompt_status
]))

In [12]:
from pathlib import Path
from datetime import datetime

def export_chat(filename=None):
    if not chat_history:
        print("No conversation to export.")
        return None

    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"chat_history_{timestamp}.txt"

    lines = []

    for message in chat_history:
        role = message["role"].upper()
        content = message["content"]
        lines.append(f"{role}\n{content}\n")

    path = Path(filename)
    path.write_text("\n".join(lines), encoding="utf-8")

    print(f"Conversation saved to: {path.resolve()}")
    return path

# Run this whenever you want to save the current conversation.
export_chat()

No conversation to export.
